Ideas for data analysis:
* Total distance
* Total time
* Percentage of distance in each sport type
* Percentage of time in each sport type
* Number of activities per month over time
* Distance per month over time
* Time per month over time

In [9]:
import pandas as pd
from numpy import float64 as np_float64, int64 as np_int64
import json

In [32]:
def DT_TO_TS(dt): return int(dt.timestamp()) * 1000

In [10]:
def read_datastore(file_name):
    try:
        with open(file_name, 'r') as file:
            return json.load(file)
    except FileNotFoundError:
        print(f'Error reading datastore: File not found: {file_name}')
        return None
    except json.JSONDecodeError:
        print(f'Error reading datastore: Invalid JSON format in file: {file_name}')
        return None

In [11]:
data = read_datastore('data.json')['data']['activities']
df = pd.DataFrame(data)
df

,resource_state,athlete,name,distance,moving_time,elapsed_time,total_elevation_gain,type,sport_type,id,...,total_photo_count,has_kudoed,device_name,average_cadence,average_heartrate,max_heartrate,elev_high,elev_low,upload_id_str,workout_type
0,2,"{'id': 120371207, 'resource_state': 1}",Walk 423TH,3218.7,2055,2055,0.0,Walk,Walk,17002614995,...,2,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,"{'id': 120371207, 'resource_state': 1}",Walk 422,742.2,699,808,0.0,Walk,Walk,16962643635,...,0,False,Samsung Health,115.1,101.3,121.0,26.2,24.5,18056065871,NaN
2,2,"{'id': 120371207, 'resource_state': 1}",Walk 421,446.0,375,375,0.0,Walk,Walk,16959144820,...,0,False,Samsung Health,118.5,110.5,118.0,33.3,31.7,18052538364,NaN
3,2,"{'id': 120371207, 'resource_state': 1}",Stair Stepper 29H,4055.6,1441,1441,0.0,StairStepper,StairStepper,16963326283,...,2,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.0
4,2,"{'id': 120371207, 'resource_state': 1}",Walk 420,1622.6,1120,1120,13.4,Walk,Walk,16955608989,...,0,False,Samsung Health,122.9,119.3,129.0,32.4,18.8,18048972528,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2,"{'id': 120371207, 'resource_state': 1}",Warmup Run 367T,836.9,377,377,0.0,Run,Run,15775263133,...,0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
196,2,"{'id': 120371207, 'resource_state': 1}",Cycling 25,8095.0,1086,1086,0.0,EBikeRide,EBikeRide,15758622406,...,2,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
197,2,"{'id': 120371207, 'resource_state': 1}",Elliptical 26RX,3234.8,820,820,0.0,Workout,Elliptical,15745438931,...,2,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
198,2,"{'id': 120371207, 'resource_state': 1}",Run 366T,5005.1,1836,1836,0.0,Run,Run,15745167246,...,2,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0


In [12]:
if "start_date_dt" not in df.columns:
    df["start_date_dt"] = pd.to_datetime(df["start_date"])
df["start_date_dt"]

0     2026-01-09 20:43:00+00:00
1     2026-01-06 23:48:47+00:00
2     2026-01-06 17:57:56+00:00
3     2026-01-06 16:47:00+00:00
4     2026-01-06 12:06:30+00:00
                 ...           
195   2025-09-10 21:21:00+00:00
196   2025-09-09 23:42:00+00:00
197   2025-09-08 20:51:00+00:00
198   2025-09-08 20:17:00+00:00
199   2025-09-08 20:10:00+00:00
Name: start_date_dt, Length: 200, dtype: datetime64[ns, UTC]

In [44]:
df.groupby(by="sport_type")["sport_type"].count()

sport_type
EBikeRide         8
Elliptical        8
Run             104
StairStepper      6
Walk             68
Workout           6
Name: sport_type, dtype: int64

In [33]:
series = df.groupby(pd.Grouper(key="start_date_dt", freq="W-MON", label="left", closed="left"))["id"].count().rename(index=DT_TO_TS)
series

start_date_dt
1757289600000    15
1757894400000    12
1758499200000    13
1759104000000    17
1759708800000    15
1760313600000    14
1760918400000    15
1761523200000    14
1762128000000    14
1762732800000    11
1763337600000    10
1763942400000     6
1764547200000    15
1765152000000    13
1765756800000     8
1766361600000     1
1766966400000     2
1767571200000     5
Name: id, dtype: int64

In [19]:
series[[1757289600000, 1761523200000]] = 0
series

start_date_dt
1757289600000        0.0
1757894400000    42546.3
1758499200000    36624.3
1759104000000    55769.1
1759708800000    43487.3
1760313600000    32837.2
1760918400000    41430.4
1761523200000        0.0
1762128000000    38519.7
1762732800000    36648.1
1763337600000    20879.8
1763942400000    24735.8
1764547200000    43790.5
1765152000000    31076.8
1765756800000    27755.7
1766361600000     6839.7
1766966400000    12391.9
1767571200000    10085.1
Name: distance, dtype: float64

In [31]:
series[series != 0].to_dict()

{1757289600000: 55920.2,
 1757894400000: 42546.3,
 1758499200000: 36624.3,
 1759104000000: 55769.1,
 1759708800000: 43487.3,
 1760313600000: 32837.2,
 1760918400000: 41430.4,
 1761523200000: 37672.5,
 1762128000000: 38519.700000000004,
 1762732800000: 36648.1,
 1763337600000: 20879.8,
 1763942400000: 24735.8,
 1764547200000: 43790.5,
 1765152000000: 31076.800000000003,
 1765756800000: 27755.7,
 1766361600000: 6839.7,
 1766966400000: 12391.9,
 1767571200000: 10085.099999999999}